# Atividade Prática 11 - Lidando com Dados do Mundo Real

Esse _notebook_ é reponsável por recomendar as músicas com base dos dados obtidos.

Os dados foram carregados do arquivo que possui as informações extraídas e transformadas agrupadas em um só arquivo csv.

In [1]:
import pandas as pd

PATH_DIRETORIO = 'dados/bkp/final_08/'

PATH_DADOS_MUSICAS = PATH_DIRETORIO + 'dados_musicas.csv'

dados_musicas = pd.read_csv(PATH_DADOS_MUSICAS, sep=';', decimal='.', index_col='id_musica')

dados_musicas.head()

,generos,musica,media_pontos,media_pts_ganho,total_pts_calc,electro,electronic,pop,alternative metal,alternative rock,...,rap rock,celtic new age,glam metal,pop metal,gothic,gothic rock,progressive metal,rap metal,future rave,hands up
id_musica,,,,,,,,,,,,,,,,,,,,,
01a02706-e024-7485-80f6-935a3c2770ff,electro|electronic|pop,36Grad,1648.285714,44.428571,0.476,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
01a02706-e024-7485-80f6-935c776327ac,alternative metal|alternative rock|heavy metal...,Be Like That,1469.000000,613.500000,0.005,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
01a02706-e024-7485-80f6-935d5d4a5e5e,alternative metal|alternative rock|arena rock|...,Here Without You,5069.066667,87.666667,6.322,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
01a02706-e024-7485-80f6-935e17059cc1,alternative rock|rock,It's Not My Time,2537.333333,191.666667,0.254,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
01a02706-e024-7485-80f6-935fd9c527ce,alternative metal|alternative rock|arena rock|...,Kryptonite,3768.916667,99.000000,3.774,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0


Normalizando os dados para serem utilizados para calcular a distância.

In [2]:
import numpy as np

def normalizar_max_min(celula, coluna, dataframe):
    serie_coluna = dataframe[coluna]
    
    return (celula - np.min(serie_coluna) ) / ( np.max(serie_coluna) - np.min(serie_coluna) ) # Funcao da normalizacao
    

calculos_dados = dados_musicas[['media_pontos', 'media_pts_ganho', 'total_pts_calc']]

# calculos_dados['n_media_pontos'] = calculos_dados['media_pontos'].apply(lambda mp: (mp - np.min(calculos_dados['media_pontos'])) / (np.max(calculos_dados['media_pontos']) - np.min(calculos_dados['media_pontos'])) )

calculos_dados['n_media_pontos'] = calculos_dados['media_pontos'].apply(
                                                lambda cell: normalizar_max_min(cell, 'media_pontos', calculos_dados) )

calculos_dados['n_media_pts_ganho'] = calculos_dados['media_pts_ganho'].apply(
                                                lambda cell: normalizar_max_min(cell, 'media_pts_ganho', calculos_dados) )

calculos_dados['n_total_pts_calc'] = calculos_dados['total_pts_calc'].apply(
                                                lambda cell: normalizar_max_min(cell, 'total_pts_calc', calculos_dados) )
    
calculos_dados.head(3) # Prefixo `n` utilizado para indicar colunas normalizadas

,media_pontos,media_pts_ganho,total_pts_calc,n_media_pontos,n_media_pts_ganho,n_total_pts_calc
id_musica,,,,,,
01a02706-e024-7485-80f6-935a3c2770ff,1648.285714,44.428571,0.476,0.079981,0.386779,0.000340
01a02706-e024-7485-80f6-935c776327ac,1469.000000,613.500000,0.005,0.060506,0.507714,0.000003
01a02706-e024-7485-80f6-935d5d4a5e5e,5069.066667,87.666667,6.322,0.451573,0.395968,0.004524


Dados foram organizados em dicionários de dicionários para análise, nele a o gênero e as pontuações normalizadas.

In [3]:

dados_dict = {}

with open(PATH_DADOS_MUSICAS, 'r') as arquivo:
    first = True
    for linha in arquivo:
        if first: # Pulando a primeira linha que é o cabeçalho
            first = False 
            continue
            
        info = linha.strip().split(';')

        id_musica = info[0]
        musica = info[2]
        media_pontos = float(info[3])
        media_pts_ganho = float(info[4])
        total_pts_calc = float(info[5])
        
        generos = info[6:] # A partir do 6 elemento é tudo genero por isso foi feita a ordenação ao agrupar os dados
        genero = list(map(int, generos)) # Converte os 0 e 1 de `str` para `int` passando a função pelo `map`

        # print(f'\n|{id_musica}|{musica}|{media_pontos}|{media_pts_ganho}|{total_pts_calc}|{genero}|')

        dados_dict[id_musica] = { # Um dicionário de dicionários para poder recuperar as informações da música
            'id_musica': id_musica,
            'musica': musica,
            #'media_pontos': media_pontos,
            #'media_pts_ganho': media_pts_ganho,
            #'total_pts_calc': total_pts_calc,
            'n_media_pontos': calculos_dados.loc[id_musica]['n_media_pontos'], # Acessa o `DataFrame` com as informações
                                                                                # normalizadas para popular o dicionário
            'n_media_pts_ganho': calculos_dados.loc[id_musica]['n_media_pts_ganho'],
            'n_total_pts_calc': calculos_dados.loc[id_musica]['n_total_pts_calc'],
            'genero': genero
        }

dados_dict['01a02706-e024-7485-80f6-935a3c2770ff']

{'id_musica': '01a02706-e024-7485-80f6-935a3c2770ff',
 'musica': '36Grad',
 'n_media_pontos': np.float64(0.07998104091797514),
 'n_media_pts_ganho': np.float64(0.3867792781852625),
 'n_total_pts_calc': np.float64(0.00033996491562070794),
 'genero': [1,
  1,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  

Calculado a diferença (distância) de uma música para a outra, essa função será utilizada para recomendar a playlist com base na música 
escolhida.

In [4]:
from scipy import spatial

def diferenca_musicas(musica_base, possivel_recomendacao):
    '''
    Quanto menor a diferença mais semelhante é a música, ou seja menor sua é a sua distância. 
    Cálcula a distância da música considerando: 
        Genêro: dado pela a distância entre os dois vetores.
        Média de Pontos, Média de Pontos ganhos, Total de Pontos: Todos definidos pela diferença do valor normalizado.
    '''

    genero_base = musica_base['genero']
    genero = possivel_recomendacao['genero']
    diferenca_genero = spatial.distance.cosine(genero_base, genero) # Calcula a distância dos vetores pelo cosseno.

    media_pontos_base = musica_base['n_media_pontos']
    media_pontos = possivel_recomendacao['n_media_pontos']
    diferenca_media_pontos = abs(media_pontos_base - media_pontos)

    media_pts_ganho_base = musica_base['n_media_pts_ganho']
    media_pts_ganho = possivel_recomendacao['n_media_pts_ganho']
    diferenca_media_pts_ganho = abs(media_pts_ganho_base - media_pts_ganho)

    total_pts_calc_base = musica_base['n_total_pts_calc']
    total_pts_calc = possivel_recomendacao['n_total_pts_calc']
    diferenca_total_pts_calc = abs(total_pts_calc_base - total_pts_calc)
    
    return diferenca_genero + diferenca_media_pontos + diferenca_media_pts_ganho + diferenca_total_pts_calc
    

print(f'\n01a02706-e024-7485-80f6-935d5d4a5e5e: {dados_dict['01a02706-e024-7485-80f6-935d5d4a5e5e']['musica']}')
print(f'01a02706-e024-7485-80f6-935e17059cc1: {dados_dict['01a02706-e024-7485-80f6-935e17059cc1']['musica']}')
print(diferenca_musicas(dados_dict['01a02706-e024-7485-80f6-935d5d4a5e5e'], dados_dict['01a02706-e024-7485-80f6-935e17059cc1']))

print(f'\n01a02706-e024-7485-80f6-935d5d4a5e5e: {dados_dict['01a02706-e024-7485-80f6-935d5d4a5e5e']['musica']}')
print(f'01a02706-e024-7485-80f6-94cf7ab2a7a2: {dados_dict['01a02706-e024-7485-80f6-94cf7ab2a7a2']['musica']}')
print(diferenca_musicas(dados_dict['01a02706-e024-7485-80f6-935d5d4a5e5e'], dados_dict['01a02706-e024-7485-80f6-94cf7ab2a7a2']))


01a02706-e024-7485-80f6-935d5d4a5e5e: Here Without You
01a02706-e024-7485-80f6-935e17059cc1: It's Not My Time
0.9363123450088683

01a02706-e024-7485-80f6-935d5d4a5e5e: Here Without You
01a02706-e024-7485-80f6-94cf7ab2a7a2: Aliens
1.5224014467236795


In [5]:
import operator

PATH_ARTISTA_MUSICA = PATH_DIRETORIO + 'kworb/transformacao/musicas_artistas.csv'

artistas_musicas = pd.read_csv(PATH_ARTISTA_MUSICA, sep=';', decimal='.')

def buscar_id_artistas(id_musica):
    return list(artistas_musicas[artistas_musicas['id_musica'] == id_musica]['id_artista'])

def encontrar_musicas_semelhantes(id_musica_escolhida, k_musicas, repetir_artista=True):
    diferencas = []

    artistas_musica_escolhida = []
    if not repetir_artista:
        artistas_musica_escolhida = buscar_id_artistas(id_musica_escolhida)
    
    for id_musica in dados_dict:

        mesmo_artista = False

        if not repetir_artista: # Verifica se algum dos artistas da música escolhida não está nas músicas que são 
                            # consideradas, caso esteja desconsidera a música.

            artistas = buscar_id_artistas(id_musica)
            artista_ambas_musicas = [artista for artista in artistas if artista in artistas_musica_escolhida]
            mesmo_artista = len(artista_ambas_musicas) > 0 # Se possuir pelo menos um artista desconsidera
        
        if id_musica != id_musica_escolhida and not mesmo_artista: # Para não considerar a música escolhida.

            diferenca = diferenca_musicas(dados_dict[id_musica_escolhida], dados_dict[id_musica]) # Calcula pelo método feito
            
            diferencas.append((id_musica, diferenca)) # Adiciona a lista de diferenças o par (tupla) do identificador
                                                        # e valor.

    diferencas.sort(key=operator.itemgetter(1)) # Ordena a tupla pelo segunda célula que é resultado da diferença

    ids_musicas_semelhantes = []
    for k in range(k_musicas):
        if k == len(diferencas):
            break
        
        ids_musicas_semelhantes.append(diferencas[k][0])

    return ids_musicas_semelhantes

encontrar_musicas_semelhantes('01a02706-e024-7485-80f6-935d5d4a5e5e', k_musicas=10)

['01a02706-e024-7485-80f6-935fd9c527ce',
 '01a02706-e024-7485-80f6-93610290ee65',
 '01a02706-e025-732e-b5ae-12e2a79183aa',
 '01a02706-e025-732e-b5ae-12e77ba679c2',
 '01a02706-e025-732e-b5ae-12df407ce716',
 '01a02706-e024-7485-80f6-935c776327ac',
 '01a02706-e024-7485-80f6-936284400805',
 '01a02706-e024-7485-80f6-941c76870791',
 '01a02706-e024-7485-80f6-944a75046939',
 '01a02706-e024-7485-80f6-9360b7fefe5a']

In [6]:


def buscar_generos(id_musica):
    '''Busca os três primeiros gêneros que estão definidos para a música informada'''
    
    genero = dados_musicas.loc[id_musica]['generos']
    genero = genero.split('|')
    generos = ''
    count = 0
    first = True
    for g in genero:
        if first:
            generos += g
            first = False
        else :
            generos += f', {g}'

        count += 1 
        if count == 2: # Mostra no máximo os 3 primeiros genêros musicais
            break

    return generos

def buscar_artistas(id_musica):
    '''Busca os Artistas que participaram da música informada'''

    artistas = list(artistas_musicas[artistas_musicas['id_musica'] == id_musica]['artista'])
    
    artistas_str = ''
    first = True
    for artista in artistas:
        if first:
            artistas_str = artista
            first = False
        else:
            artistas_str += f', {artista}'
    
    return artistas_str

def buscar_informacoes_musica(id_musica):
    '''Busca informações da música pelo id'''

    info = dados_dict[id_musica]
    
    musica = info['musica']

    artista = buscar_artistas(id_musica)

    generos = buscar_generos(id_musica)
    
    return f'{musica} by {artista} | {generos}'
    

def recomendar_playlist(id_musica_escolhida, repetir_artista=True):
    K_MUSICAS = 15
    
    musicas_semelhantes = encontrar_musicas_semelhantes(id_musica_escolhida, k_musicas=K_MUSICAS, repetir_artista=repetir_artista)

    musica_escolhida = dados_dict[id_musica_escolhida]
    
    print(f'\nAqui está a Playlist de {musica_escolhida['musica']}:\n')
    print(f'* | {buscar_informacoes_musica(id_musica_escolhida)}')
    
    k = 1
    for id_musica in musicas_semelhantes:
        print(f'{k} | {buscar_informacoes_musica(id_musica)}')
        k += 1
    print('*')

In [7]:
recomendar_playlist('01a02706-e024-7485-80f6-935d5d4a5e5e')


Aqui está a Playlist de Here Without You:

* | Here Without You by 3 Doors Down | alternative metal, alternative rock
1 | Kryptonite by 3 Doors Down | alternative metal, alternative rock
2 | Loser by 3 Doors Down, Tame Impala | alternative metal, alternative rock
3 | Lost in France by Bonnie Tyler | arena rock, ballad
4 | Total Eclipse of the Heart by Bonnie Tyler | acid rock, aor
5 | Holding Out for a Hero by Bonnie Tyler | aor, arena rock
6 | Be Like That by 3 Doors Down | alternative metal, alternative rock
7 | When I'm Gone by 3 Doors Down | alternative rock, pop
8 | Scales Are Falling by Alter Bridge | alternative metal, rock
9 | Ring My Bell by Anita Ward | club, dance
10 | Let Me Go by 3 Doors Down | alternative rock, hard rock
11 | Who Will You Follow by Evanescence | alternative metal
12 | Like Animals by BTS | alternative pop, k-pop
13 | The Final Countdown by Europe | aor, arena rock
14 | It's Not My Time by 3 Doors Down | alternative rock, rock
15 | WILDFLOWER by Billie Ei

In [8]:
recomendar_playlist('01a02706-e024-7485-80f6-94d123beb688')


Aqui está a Playlist de Butter:

* | Butter by BTS | k-pop, pop
1 | Stealer by ENHYPEN | k-pop, pop
2 | Lost Island by ENHYPEN | k-pop, pop
3 | Big Girls Don't Cry by ENHYPEN | k-pop, pop
4 | Sleep Tight by ENHYPEN | k-pop, pop
5 | As the Days Go By by Daryl Braithwaite | pop
6 | Minimum by Clueso | pop
7 | How Do I Heal by Evanescence | pop
8 | Baby Goodbye by Alle Farben | pop
9 | Immer wenn du nicht da bist by Clueso | pop
10 | Knife by ENHYPEN | k-pop, pop
11 | Die With You by Beyoncé | pop
12 | Carry You Home by Alex Warren | pop
13 | Ballon by Clueso | pop
14 | The Subway by Chappell Roan | pop
15 | We Don’t Talk Anymore by Charlie Puth | pop
*


In [9]:
recomendar_playlist('01a02706-e024-7485-80f6-935d5d4a5e5e', repetir_artista=False) # Função interessante de acordo com a 
                                                                                    # quantidade de músicas disponíveis


Aqui está a Playlist de Here Without You:

* | Here Without You by 3 Doors Down | alternative metal, alternative rock
1 | Lost in France by Bonnie Tyler | arena rock, ballad
2 | Total Eclipse of the Heart by Bonnie Tyler | acid rock, aor
3 | Holding Out for a Hero by Bonnie Tyler | aor, arena rock
4 | Scales Are Falling by Alter Bridge | alternative metal, rock
5 | Ring My Bell by Anita Ward | club, dance
6 | Who Will You Follow by Evanescence | alternative metal
7 | Like Animals by BTS | alternative pop, k-pop
8 | The Final Countdown by Europe | aor, arena rock
9 | WILDFLOWER by Billie Eilish | alternative pop, electronic
10 | Everything I Own by Bread | ballad, beat music
11 | Uninvited by Alanis Morissette | alternative rock, pop
12 | Heat of the Moment by Asia | aor, arena rock
13 | Afterlife by Evanescence | alternative metal
14 | Dream On by Aerosmith | arena rock, blues
15 | It's a Heartache by Bonnie Tyler | ballad, pop
*


In [10]:
recomendar_playlist('01a02706-e024-7485-80f6-93f04860a37d', repetir_artista=True) 


Aqui está a Playlist de Remember When:

* | Remember When by Alan Jackson | classic country, contemporary country
1 | Dust on the Bottle by David Lee Murphy, Jason Aldean | classic country, contemporary country
2 | Tennessee Whiskey by Chris Stapleton | contemporary country, country
3 | Dandelion by Ella Langley | contemporary country, country
4 | Livin’ on Borrowed Time by Breaking Rust | country
5 | Give Me Back My Hometown by Eric Church | contemporary country, country
6 | Wagon Wheel by Darius Rucker | blues, contemporary country
7 | The Fall by Cody Johnson | country
8 | The Ride by David Allan Coe | americana, classic country
9 | You Never Even Called Me by My Name by David Allan Coe | americana, classic country
10 | Jolene by Dolly Parton | alternative rock, big beat
11 | I Will Always Love You by Dolly Parton | bluegrass, classic country
12 | Be Her by Ella Langley | country, country pop
13 | American Pie by Don McLean | classic rock, contemporary folk
14 | 9 to 5 by Dolly Par